# Paper Trader - A-Share Backtest Simulation

Interactive backtest engine with step-by-step daily trading.

**Sections:**
1. Session Setup & Data Loading
2. Step-by-Step Daily Trading
3. Backtest Results Visualization
4. Performance Analysis

In [ ]:
import sys
sys.path.insert(0, 'mcp-servers/paper-trader')

from server import create_session, load_bar_data, step_session, submit_signal, submit_signals_batch
from server import get_equity_curve, get_trade_log, get_performance, list_sessions, get_session_status
from server import _engines
from models import Signal
from charts import (
    plot_equity_curve, plot_drawdown, plot_excess_returns,
    plot_kline, plot_cost_breakdown, plot_performance_card,
    display_positions_table, display_trades_table,
)
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
import json

print('All modules loaded.')

---
## 1. Session Setup & Data Loading

In [ ]:
# Configuration
START_DATE = '20250401'
END_DATE   = '20250430'
CAPITAL    = 1000000.0
UNIVERSE   = ['000001', '600519']   # Ping An Bank, Kweichow Moutai
BENCHMARK  = 'sh000300'              # CSI 300

In [ ]:
# Create session
result = create_session(
    start_date=START_DATE,
    end_date=END_DATE,
    name='notebook_test',
    strategy='interactive',
    initial_capital=CAPITAL,
    universe=UNIVERSE,
    benchmark=BENCHMARK,
)
SESSION_ID = result[0]['session_id']
print(f'Session created: {SESSION_ID}')
print(f'  Period: {START_DATE} ~ {END_DATE}')
print(f'  Capital: ¥{CAPITAL:,.0f}')
print(f'  Universe: {UNIVERSE}')

In [ ]:
# Load market data via MCP tools
# NOTE: Run this after akshare MCP server is available
# If running offline, use the hardcoded sample data below

USE_MCP = False  # Set True if akshare MCP server is running

if USE_MCP:
    # Fetch from akshare MCP
    from mcp import ClientSession
    # ... MCP client call ...
    pass
else:
    # Sample data: April 2025
    def _norm(d): return d.replace('-', '')
    
    RAW_000001 = [
        {'日期': '2025-04-01', '开盘': 10.672, '收盘': 10.672, '最高': 10.702, '最低': 10.622, '成交量': 68.147},
        {'日期': '2025-04-02', '开盘': 10.652, '收盘': 10.772, '最高': 10.812, '最低': 10.652, '成交量': 93.264},
        {'日期': '2025-04-03', '开盘': 10.712, '收盘': 10.742, '最高': 10.792, '最低': 10.702, '成交量': 64.391},
        {'日期': '2025-04-07', '开盘': 10.402, '收盘': 10.102, '最高': 10.452, '最低': 9.882, '成交量': 254.556},
        {'日期': '2025-04-08', '开盘': 10.082, '收盘': 10.222, '最高': 10.252, '最低': 10.062, '成交量': 146.399},
        {'日期': '2025-04-09', '开盘': 10.132, '收盘': 10.202, '最高': 10.232, '最低': 10.062, '成交量': 106.439},
        {'日期': '2025-04-10', '开盘': 10.262, '收盘': 10.302, '最高': 10.332, '最低': 10.222, '成交量': 86.655},
        {'日期': '2025-04-11', '开盘': 10.272, '收盘': 10.292, '最高': 10.302, '最低': 10.232, '成交量': 58.388},
        {'日期': '2025-04-14', '开盘': 10.352, '收盘': 10.332, '最高': 10.442, '最低': 10.322, '成交量': 78.599},
        {'日期': '2025-04-15', '开盘': 10.322, '收盘': 10.352, '最高': 10.372, '最低': 10.302, '成交量': 72.275},
        {'日期': '2025-04-16', '开盘': 10.332, '收盘': 10.402, '最高': 10.412, '最低': 10.312, '成交量': 84.282},
        {'日期': '2025-04-17', '开盘': 10.352, '收盘': 10.462, '最高': 10.482, '最低': 10.332, '成交量': 82.392},
        {'日期': '2025-04-18', '开盘': 10.442, '收盘': 10.582, '最高': 10.592, '最低': 10.432, '成交量': 72.710},
        {'日期': '2025-04-21', '开盘': 10.402, '收盘': 10.422, '最高': 10.532, '最低': 10.372, '成交量': 111.418},
        {'日期': '2025-04-22', '开盘': 10.422, '收盘': 10.442, '最高': 10.462, '最低': 10.382, '成交量': 83.113},
        {'日期': '2025-04-23', '开盘': 10.442, '收盘': 10.412, '最高': 10.452, '最低': 10.372, '成交量': 58.638},
        {'日期': '2025-04-24', '开盘': 10.402, '收盘': 10.432, '最高': 10.462, '最低': 10.392, '成交量': 68.926},
        {'日期': '2025-04-25', '开盘': 10.442, '收盘': 10.412, '最高': 10.452, '最低': 10.392, '成交量': 63.755},
        {'日期': '2025-04-28', '开盘': 10.402, '收盘': 10.402, '最高': 10.442, '最低': 10.362, '成交量': 64.202},
        {'日期': '2025-04-29', '开盘': 10.402, '收盘': 10.382, '最高': 10.422, '最低': 10.352, '成交量': 64.905},
        {'日期': '2025-04-30', '开盘': 10.362, '收盘': 10.312, '最高': 10.372, '最低': 10.292, '成交量': 86.984},
    ]
    RAW_600519 = [
        {'日期': '2025-04-01', '开盘': 1512.443, '收盘': 1504.463, '最高': 1518.333, '最低': 1500.653, '成交量': 1.856},
        {'日期': '2025-04-02', '开盘': 1506.443, '收盘': 1497.463, '最高': 1516.243, '最低': 1493.943, '成交量': 2.167},
        {'日期': '2025-04-03', '开盘': 1478.443, '收盘': 1517.323, '最高': 1534.443, '最低': 1477.453, '成交量': 3.548},
        {'日期': '2025-04-07', '开盘': 1468.453, '收盘': 1448.443, '最高': 1485.293, '最低': 1410.443, '成交量': 10.195},
        {'日期': '2025-04-08', '开盘': 1461.443, '收盘': 1493.443, '最高': 1493.443, '最低': 1443.443, '成交量': 7.388},
        {'日期': '2025-04-09', '开盘': 1473.463, '收盘': 1489.483, '最高': 1505.243, '最低': 1468.993, '成交量': 5.563},
        {'日期': '2025-04-10', '开盘': 1499.433, '收盘': 1497.443, '最高': 1503.443, '最低': 1476.443, '成交量': 3.862},
        {'日期': '2025-04-11', '开盘': 1528.413, '收盘': 1517.423, '最高': 1528.413, '最低': 1493.443, '成交量': 3.263},
        {'日期': '2025-04-14', '开盘': 1509.413, '收盘': 1500.433, '最高': 1514.443, '最低': 1499.973, '成交量': 2.171},
        {'日期': '2025-04-15', '开盘': 1500.443, '收盘': 1506.443, '最高': 1513.443, '最低': 1493.443, '成交量': 2.149},
        {'日期': '2025-04-16', '开盘': 1500.443, '收盘': 1507.613, '最高': 1524.443, '最低': 1485.443, '成交量': 3.116},
        {'日期': '2025-04-17', '开盘': 1502.443, '收盘': 1518.443, '最高': 1524.943, '最低': 1498.433, '成交量': 2.385},
        {'日期': '2025-04-18', '开盘': 1514.443, '收盘': 1514.383, '最高': 1523.443, '最低': 1504.443, '成交量': 2.030},
        {'日期': '2025-04-21', '开盘': 1513.943, '收盘': 1499.443, '最高': 1513.943, '最低': 1499.443, '成交量': 1.806},
        {'日期': '2025-04-22', '开盘': 1498.443, '收盘': 1497.243, '最高': 1504.743, '最低': 1491.653, '成交量': 1.843},
        {'日期': '2025-04-23', '开盘': 1507.443, '收盘': 1500.443, '最高': 1507.663, '最低': 1493.443, '成交量': 1.867},
        {'日期': '2025-04-24', '开盘': 1500.443, '收盘': 1500.693, '最高': 1509.123, '最低': 1497.423, '成交量': 1.487},
        {'日期': '2025-04-25', '开盘': 1505.543, '收盘': 1498.443, '最高': 1509.643, '最低': 1498.443, '成交量': 1.477},
        {'日期': '2025-04-28', '开盘': 1500.443, '收盘': 1498.443, '最高': 1503.443, '最低': 1495.043, '成交量': 1.466},
        {'日期': '2025-04-29', '开盘': 1498.443, '收盘': 1492.443, '最高': 1500.983, '最低': 1480.463, '成交量': 1.892},
        {'日期': '2025-04-30', '开盘': 1498.433, '收盘': 1495.443, '最高': 1515.103, '最低': 1494.743, '成交量': 2.575},
    ]
    RAW_BENCHMARK = [
        {'date': '2025-04-01', 'close': 3887.684},
        {'date': '2025-04-02', 'close': 3884.386},
        {'date': '2025-04-03', 'close': 3861.503},
        {'date': '2025-04-07', 'close': 3589.441},
        {'date': '2025-04-08', 'close': 3650.759},
        {'date': '2025-04-09', 'close': 3686.794},
        {'date': '2025-04-10', 'close': 3735.115},
        {'date': '2025-04-11', 'close': 3750.517},
        {'date': '2025-04-14', 'close': 3759.142},
        {'date': '2025-04-15', 'close': 3761.235},
        {'date': '2025-04-16', 'close': 3772.820},
        {'date': '2025-04-17', 'close': 3772.222},
        {'date': '2025-04-18', 'close': 3772.523},
        {'date': '2025-04-21', 'close': 3784.881},
        {'date': '2025-04-22', 'close': 3783.952},
        {'date': '2025-04-23', 'close': 3786.882},
        {'date': '2025-04-24', 'close': 3784.357},
        {'date': '2025-04-25', 'close': 3786.994},
        {'date': '2025-04-28', 'close': 3781.619},
        {'date': '2025-04-29', 'close': 3775.077},
        {'date': '2025-04-30', 'close': 3770.571},
    ]
    
    stock_data = {
        '000001': [{**r, '日期': _norm(r['日期'])} for r in RAW_000001],
        '600519': [{**r, '日期': _norm(r['日期'])} for r in RAW_600519],
    }
    benchmark_data = [{'日期': _norm(r['date']), '收盘': r['close']} for r in RAW_BENCHMARK]

load_result = load_bar_data(SESSION_ID, UNIVERSE, stock_data, benchmark_data)
print(f'Loaded: {load_result}')

---
## 2. Step-by-Step Daily Trading

Click **Step** to advance one trading day. Use the controls below to submit signals.

In [ ]:
# State
_step_state = {'done': False}

# Widgets
step_btn = widgets.Button(description='Step', button_style='info', layout=widgets.Layout(width='100px'))
stock_dropdown = widgets.Dropdown(options=UNIVERSE, description='Stock:')
direction_dropdown = widgets.Dropdown(options=['buy', 'sell'], description='Direction:')
weight_slider = widgets.FloatSlider(value=0.3, min=0.05, max=1.0, step=0.05, description='Weight:')
signal_btn = widgets.Button(description='Submit Signal', button_style='warning')
output = widgets.Output()

def on_step(b):
    with output:
        clear_output(wait=True)
        if _step_state['done']:
            print('Session completed!')
            return
        r = step_session(SESSION_ID)
        if not r or 'error' in r[0]:
            print(f'Error: {r}')
            return
        r = r[0]
        if r['status'] == 'completed':
            _step_state['done'] = True
            print(f'Session completed! Final NAV: ¥{r["final_nav"]:,.2f}')
            return
        print(f'===== Day {r["day_index"]}/{r["total_days"]} | {r["trade_date"]} =====')
        print(f'NAV: ¥{r["nav"]:,.2f}  |  Cash: ¥{r["cash"]:,.2f}  |  Remaining: {r["remaining_days"]} days')
        print()
        print('Market Data:')
        for code, bar in r.get('market_data', {}).items():
            chg = (bar['close'] - bar['open']) / bar['open'] * 100
            arrow = '+' if chg >= 0 else ''
            print(f'  {code}: Open={bar["open"]:.3f} Close={bar["close"]:.3f} ({arrow}{chg:.2f}%)')
        if r.get('positions'):
            print()
            print('Positions:')
            display(display_positions_table(r['positions']))

def on_signal(b):
    with output:
        engine = _engines.get(SESSION_ID)
        if not engine:
            print('No active session')
            return
        # Get current date from engine
        if not engine._trading_days or engine._day_index == 0:
            print('Step first to get a trading date')
            return
        current_date = engine._trading_days[min(engine._day_index, len(engine._trading_days)) - 1]
        result = submit_signal(
            SESSION_ID,
            signal_date=current_date,
            stock_code=stock_dropdown.value,
            direction=direction_dropdown.value,
            target_weight=weight_slider.value if direction_dropdown.value == 'buy' else 0,
        )
        print(f'Signal submitted: {result[0]}')

step_btn.on_click(on_step)
signal_btn.on_click(on_signal)

display(widgets.HBox([step_btn, stock_dropdown, direction_dropdown, weight_slider, signal_btn]))
display(output)

In [ ]:
# Quick: run all remaining steps in batch
print('Running all remaining days...')
engine = _engines.get(SESSION_ID)
if engine:
    while engine.config.status != 'completed':
        r = step_session(SESSION_ID)
        if not r or 'error' in r[0]:
            break
        r = r[0]
        if r.get('status') == 'completed':
            print(f'Completed! Final NAV: ¥{r["final_nav"]:,.2f}')
            break
        print(f'Day {r["day_index"]} {r["trade_date"]} NAV=¥{r["nav"]:,.2f}')
else:
    print('No engine found. Run step at least once first.')

---
## 3. Backtest Results Visualization

In [ ]:
# Load results
curve = get_equity_curve(SESSION_ID)
trades = get_trade_log(SESSION_ID)
perf = get_performance(SESSION_ID)

if curve and 'error' not in curve[0]:
    plot_equity_curve(curve).show()
    plot_drawdown(curve).show()
    plot_excess_returns(curve).show()
else:
    print('No equity curve data. Run the backtest first.')

In [ ]:
# K-line chart with trade markers for 000001
engine = _engines.get(SESSION_ID)
if engine and engine.bar_data.get('000001'):
    bars = [{'date': d, 'open': b.open, 'high': b.high, 'low': b.low, 'close': b.close, 'volume': b.volume}
            for d, b in sorted(engine.bar_data['000001'].items())]
    trade_markers = [t for t in trades if t.get('stock_code') == '000001'] if 'error' not in trades[0] else []
    plot_kline(bars, trade_markers, title='000001 Ping An Bank').show()
else:
    print('No bar data loaded.')

---
## 4. Performance Analysis

In [ ]:
# Performance metrics card
if perf and 'error' not in perf[0]:
    plot_performance_card(perf[0])
    plot_cost_breakdown(trades if trades and 'error' not in trades[0] else []).show()
else:
    print('No performance data.')

In [ ]:
# Trade log table
if trades and 'error' not in trades[0]:
    display(display_trades_table(trades))
else:
    print('No trades recorded.')